Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital\
**Tecnología Digital VI: Inteligencia Artificial**

# **Introducción al procesamiento de lenguaje natural (NLP)**

### **Tokenización**

In [6]:
import os
import re
import random
from nltk import download
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize

download('punkt') # Tokenizador utilizado por sent_tokenize.

[nltk_data] Downloading package punkt to /home/valenb/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [7]:
TRAIN_PATH = './td6-p10-d-datos-imdb/corpus/train'
TEST_PATH = './td6-p10-d-datos-imdb/corpus/test'

In [8]:
def load_imdb_corpus(path):
    '''Itera sobre el corpus y devuelve un diccionario con el ID, el puntaje y
       el texto "crudo".'''

    reviews = os.listdir(path)
    random.Random(1234).shuffle(reviews)

    for review in reviews:
        movie, rank = review.split('.')[0].split('_')
        with open(path + '/' + review, encoding = 'utf-8') as review_file:
            raw_text = review_file.read()
        yield({'id': review,
               'rank': rank,
               'raw_text': raw_text})

In [9]:
def tokenize(raw_text):
    '''Realiza la tokenización.'''

    # Primero, adapta el tokenizador por defecto, haciendo limpieza de los
    # siguientes caracteres.
    raw_text = raw_text.replace("\'", "'")
    raw_text = raw_text.replace('<br /><br />', '\n')
    raw_text = raw_text.replace('/', ' / ')

    sentences = sent_tokenize(raw_text)
    tokens = [token for sentence in sentences for token in word_tokenize(sentence)] # Generación de una lista por comprensión anidada.
    tokens = [token.lower() for token in tokens if re.compile('[A-Za-z]').search(token[0])] # re.compile para que cada token tenga, al menos, una letra.

    return(tokens)

In [10]:
imdb_tr = load_imdb_corpus(TRAIN_PATH)

In [11]:
review = next(imdb_tr)
print(review)

{'id': '290_2.txt', 'rank': '2', 'raw_text': "Killer Tomatoes movies have this special kind of humor - you either love it or hate it. I personally like it, but in this fourth movie the feeling is gone. The tomatoes aren't the same, jokes are lame, even the actors aren't as funny. Because that's the only thing this kind of movies are supposed to be - funny.<br /><br />So now following the plot made to laugh, is annoying. They really shouldn't have done the fourth part to the Killer Tomatoes trilogy."}


In [12]:
tokens = tokenize(review['raw_text'])
print(tokens)
print(tokens[0])
print(tokens[1])

['killer', 'tomatoes', 'movies', 'have', 'this', 'special', 'kind', 'of', 'humor', 'you', 'either', 'love', 'it', 'or', 'hate', 'it', 'i', 'personally', 'like', 'it', 'but', 'in', 'this', 'fourth', 'movie', 'the', 'feeling', 'is', 'gone', 'the', 'tomatoes', 'are', "n't", 'the', 'same', 'jokes', 'are', 'lame', 'even', 'the', 'actors', 'are', "n't", 'as', 'funny', 'because', 'that', 'the', 'only', 'thing', 'this', 'kind', 'of', 'movies', 'are', 'supposed', 'to', 'be', 'funny', 'so', 'now', 'following', 'the', 'plot', 'made', 'to', 'laugh', 'is', 'annoying', 'they', 'really', 'should', "n't", 'have', 'done', 'the', 'fourth', 'part', 'to', 'the', 'killer', 'tomatoes', 'trilogy']
killer
tomatoes


### **Bag-of-words model**

##### **Ilustración**

In [13]:
from collections import Counter

In [14]:
def text_to_bow(tokens):
    return(Counter(tokens))
    # Counter es un tipo de dato. Se le pasa un iterable y devuelve cuántas
    # veces aparece cada uno de sus elementos.

In [15]:
print(text_to_bow(tokens))

Counter({'the': 8, 'are': 4, 'tomatoes': 3, 'this': 3, 'it': 3, "n't": 3, 'to': 3, 'killer': 2, 'movies': 2, 'have': 2, 'kind': 2, 'of': 2, 'fourth': 2, 'is': 2, 'funny': 2, 'special': 1, 'humor': 1, 'you': 1, 'either': 1, 'love': 1, 'or': 1, 'hate': 1, 'i': 1, 'personally': 1, 'like': 1, 'but': 1, 'in': 1, 'movie': 1, 'feeling': 1, 'gone': 1, 'same': 1, 'jokes': 1, 'lame': 1, 'even': 1, 'actors': 1, 'as': 1, 'because': 1, 'that': 1, 'only': 1, 'thing': 1, 'supposed': 1, 'be': 1, 'so': 1, 'now': 1, 'following': 1, 'plot': 1, 'made': 1, 'laugh': 1, 'annoying': 1, 'they': 1, 'really': 1, 'should': 1, 'done': 1, 'part': 1, 'trilogy': 1})


##### **Matriz documento-término**

In [16]:
from tqdm import tqdm

In [17]:
def tokenize_all(corpus_iterator, size = None):
    '''Procesa todas las reseñas.'''

    all_reviews = []

    for i, review in tqdm(enumerate(corpus_iterator)): # tqdm muestra grado de avance.
        tokens = tokenize(review['raw_text'])
        tokens_count = text_to_bow(tokens)

        all_reviews.append({'id': review['id'],
                            'rank': review['rank'],
                            'tokens': tokens,
                            'tokens_count': tokens_count})

        if size and (i + 1) == size:
            break

    return(all_reviews)

In [18]:
review_tokens = tokenize_all(imdb_tr)

24999it [00:32, 759.86it/s]


In [19]:
def get_freq_tokens(tokenized_data, min_freq):
    '''Calcula la frecuencia de las palabras a lo largo de un corpus y se queda
       con aquellas que aparecen al menos tantas veces como min_freq.'''

    tokens_freqs = {}

    for document in tokenized_data:
        for token in document['tokens_count']:
            tokens_freqs[token] = tokens_freqs.get(token, 0) + document['tokens_count'][token]

    frequent_tokens = set([t for t in tokens_freqs if tokens_freqs[t] >= min_freq])

    return(frequent_tokens)

In [20]:
freq_tokens = get_freq_tokens(review_tokens, 5) # Nos quedamos con palabras que aparecen, al menos, 5 veces.

In [21]:
print(len(freq_tokens))

29584


In [22]:
import pandas as pd

In [23]:
def corpus_to_bow(tokenized_data, freq_tokens):
    '''Estructura los datos como un DataFrame de Pandas.'''

    filtered_tokens = []
    for document in tokenized_data:
        filtered_tokens.append({token:document['tokens_count'][token] for token in document['tokens_count'] if token in freq_tokens})

    ids = [document['id'] for document in tokenized_data]
    bag_of_words = pd.DataFrame(filtered_tokens, index = ids)
    bag_of_words = bag_of_words.reindex(columns = sorted(freq_tokens)).fillna(0)

    ranks = pd.Series([document['rank'] for document in tokenized_data], index = ids)

    return(bag_of_words, ranks)

In [24]:
bag_of_words, ranks = corpus_to_bow(review_tokens, freq_tokens)

In [25]:
print(bag_of_words.head())

               a   a+   a-  a-list  a-team   a.  a.i  a.k.a  a.m.  aaa  ...  \
4992_4.txt   1.0  0.0  0.0     0.0     1.0  0.0  0.0    0.0   0.0  0.0  ...   
4016_4.txt   5.0  0.0  0.0     0.0     0.0  0.0  0.0    0.0   0.0  0.0  ...   
9594_1.txt   5.0  0.0  0.0     0.0     0.0  0.0  0.0    0.0   0.0  0.0  ...   
4573_1.txt  29.0  0.0  0.0     0.0     0.0  0.0  0.0    0.0   0.0  0.0  ...   
8296_4.txt   8.0  0.0  0.0     0.0     0.0  0.0  0.0    0.0   0.0  0.0  ...   

            zorak  zorro   zp   zu  zucco  zucker  zuckerman  zulu  zuniga  \
4992_4.txt    0.0    0.0  0.0  0.0    0.0     0.0        0.0   0.0     0.0   
4016_4.txt    0.0    0.0  0.0  0.0    0.0     0.0        0.0   0.0     0.0   
9594_1.txt    0.0    0.0  0.0  0.0    0.0     0.0        0.0   0.0     0.0   
4573_1.txt    0.0    0.0  0.0  0.0    0.0     0.0        0.0   0.0     0.0   
8296_4.txt    0.0    0.0  0.0  0.0    0.0     0.0        0.0   0.0     0.0   

            zwick  
4992_4.txt    0.0  
4016_4.txt    0.

In [26]:
print(ranks.head())

4992_4.txt    4
4016_4.txt    4
9594_1.txt    1
4573_1.txt    1
8296_4.txt    4
dtype: object


In [27]:
# Ranking de palabras, en función de su frecuencia.
bag_of_words.sum(0).sort_values(ascending = False)

the          334844.0
and          163660.0
a            162314.0
of           145432.0
to           135203.0
               ...   
tingling          5.0
tinged            5.0
timm              5.0
timetable         5.0
agonized          5.0
Length: 29584, dtype: float64

In [28]:
import pickle

In [29]:
RECALCULATE_TOKENS = True
# RECALCULATE_TOKENS = False

In [30]:
if RECALCULATE_TOKENS:
    imdb_tr = load_imdb_corpus(TRAIN_PATH)
    imdb_ts = load_imdb_corpus(TEST_PATH)

    review_tokens_tr = tokenize_all(imdb_tr)
    review_tokens_ts = tokenize_all(imdb_ts)

    with open('td6-p10-e-review-tokens.p', 'wb') as file:
        pickle.dump([review_tokens_tr, review_tokens_ts], file)
else:
    with open('td6-p10-e-review-tokens.p', 'rb') as file:
        review_tokens_tr, review_tokens_ts = pickle.load(file)

25000it [00:32, 764.91it/s]
25000it [00:32, 771.87it/s]


In [31]:
freq_tokens = get_freq_tokens(review_tokens_tr, 100)
# ¡Aclaración importante! Lo corremos sólo con los datos de entrenamiento para evitar hacer data leakage, al calcular las frecuencias de las palabras.

In [32]:
X_tr, y_tr = corpus_to_bow(review_tokens_tr, freq_tokens)
X_ts, y_ts = corpus_to_bow(review_tokens_ts, freq_tokens)

In [33]:
print(X_tr.head())

               a  abandoned  abc  abilities  ability  able  about  above  \
290_2.txt    0.0        0.0  0.0        0.0      0.0   0.0    0.0    0.0   
4992_4.txt   1.0        0.0  0.0        0.0      0.0   0.0    0.0    0.0   
4016_4.txt   5.0        0.0  0.0        0.0      0.0   0.0    0.0    0.0   
9594_1.txt   5.0        0.0  0.0        0.0      0.0   0.0    2.0    0.0   
4573_1.txt  29.0        0.0  0.0        0.0      0.0   0.0   10.0    0.0   

            absence  absolute  ...  you  young  younger  your  yourself  \
290_2.txt       0.0       0.0  ...  1.0    0.0      0.0   0.0       0.0   
4992_4.txt      0.0       0.0  ...  1.0    0.0      0.0   0.0       0.0   
4016_4.txt      0.0       0.0  ...  0.0    0.0      0.0   0.0       0.0   
9594_1.txt      0.0       0.0  ...  3.0    0.0      0.0   0.0       0.0   
4573_1.txt      0.0       0.0  ...  9.0    2.0      0.0   1.0       0.0   

            youth  zero  zombie  zombies  zone  
290_2.txt     0.0   0.0     0.0      0.0   

In [34]:
print(X_ts.head())

                a  abandoned  abc  abilities  ability  able  about  above  \
2765_1.txt    5.0        0.0  0.0        0.0      0.0   0.0    0.0    0.0   
8143_4.txt    4.0        0.0  0.0        0.0      0.0   0.0    0.0    0.0   
11537_3.txt  19.0        0.0  0.0        0.0      0.0   0.0    0.0    0.0   
4161_9.txt   17.0        0.0  0.0        0.0      0.0   1.0    3.0    0.0   
1157_1.txt   10.0        0.0  0.0        0.0      0.0   0.0    1.0    0.0   

             absence  absolute  ...  you  young  younger  your  yourself  \
2765_1.txt       0.0       0.0  ...  1.0    0.0      0.0   1.0       0.0   
8143_4.txt       0.0       0.0  ...  1.0    0.0      0.0   0.0       0.0   
11537_3.txt      0.0       0.0  ...  0.0    0.0      3.0   0.0       0.0   
4161_9.txt       0.0       0.0  ...  0.0    1.0      1.0   0.0       0.0   
1157_1.txt       0.0       0.0  ...  2.0    0.0      0.0   0.0       0.0   

             youth  zero  zombie  zombies  zone  
2765_1.txt     0.0   0.0     0

In [35]:
print(y_tr) # ¡Exploramos sólo las de entrenamiento!

290_2.txt      2
4992_4.txt     4
4016_4.txt     4
9594_1.txt     1
4573_1.txt     1
              ..
10763_8.txt    8
1965_7.txt     7
2918_9.txt     9
1745_7.txt     7
10233_1.txt    1
Length: 25000, dtype: object


In [36]:
print(y_ts)

2765_1.txt       1
8143_4.txt       4
11537_3.txt      3
4161_9.txt       9
1157_1.txt       1
                ..
12214_10.txt    10
646_1.txt        1
1487_3.txt       3
6376_2.txt       2
1923_9.txt       9
Length: 25000, dtype: object


### **Similitud coseno y normalización TF-IDF**

In [37]:
from sklearn.feature_extraction.text import TfidfTransformer

In [38]:
# Configuramos la transformación a TF-IDF.
transformer = TfidfTransformer(sublinear_tf = True)
# sublinear_tf = True para reemplazar tf with 1 + log(tf).
# Documentación oficial: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html

In [39]:
X_tr = pd.DataFrame(transformer.fit_transform(X_tr).todense(),
                    columns = X_tr.columns,
                    index = X_tr.index)
# fit_transform
    # fit aprende los parámetros del modelo.
    # transform hace la transformación.

In [40]:
print(X_tr.head())

                   a  abandoned  abc  abilities  ability  able     about  \
290_2.txt   0.000000        0.0  0.0        0.0      0.0   0.0  0.000000   
4992_4.txt  0.030514        0.0  0.0        0.0      0.0   0.0  0.000000   
4016_4.txt  0.085180        0.0  0.0        0.0      0.0   0.0  0.000000   
9594_1.txt  0.056285        0.0  0.0        0.0      0.0   0.0  0.066038   
4573_1.txt  0.048547        0.0  0.0        0.0      0.0   0.0  0.066383   

            above  absence  absolute  ...       you     young  younger  \
290_2.txt     0.0      0.0       0.0  ...  0.060204  0.000000      0.0   
4992_4.txt    0.0      0.0       0.0  ...  0.047499  0.000000      0.0   
4016_4.txt    0.0      0.0       0.0  ...  0.000000  0.000000      0.0   
9594_1.txt    0.0      0.0       0.0  ...  0.070463  0.000000      0.0   
4573_1.txt    0.0      0.0       0.0  ...  0.055323  0.058708      0.0   

                your  yourself  youth      zero  zombie  zombies  zone  
290_2.txt   0.000000     

In [41]:
X_ts = pd.DataFrame(transformer.transform(X_ts).todense(),
                    columns = X_ts.columns,
                    index = X_ts.index)
# Importante: acá no hay fit.
# Como es test, se aplica sólo la transformación, con los parámetros aprendidos
# en entrenamiento, para evitar data leakage.

In [42]:
print(X_ts.head())

                    a  abandoned  abc  abilities  ability      able     about  \
2765_1.txt   0.065393        0.0  0.0        0.0      0.0  0.000000  0.000000   
8143_4.txt   0.056039        0.0  0.0        0.0      0.0  0.000000  0.000000   
11537_3.txt  0.050618        0.0  0.0        0.0      0.0  0.000000  0.000000   
4161_9.txt   0.050958        0.0  0.0        0.0      0.0  0.052723  0.050448   
1157_1.txt   0.062124        0.0  0.0        0.0      0.0  0.000000  0.034014   

             above  absence  absolute  ...       you     young   younger  \
2765_1.txt     0.0      0.0       0.0  ...  0.039010  0.000000  0.000000   
8143_4.txt     0.0      0.0       0.0  ...  0.036555  0.000000  0.000000   
11537_3.txt    0.0      0.0       0.0  ...  0.000000  0.000000  0.130610   
4161_9.txt     0.0      0.0       0.0  ...  0.000000  0.041467  0.064473   
1157_1.txt     0.0      0.0       0.0  ...  0.049577  0.000000  0.000000   

                 your  yourself  youth  zero    zombie  

In [43]:
import numpy as np

In [44]:
review = '4_10.txt' # Una reseña de test.
review_loc = np.where(X_ts.index == review)[0]

In [45]:
print(review_loc)

[9731]


In [46]:
cos_sim = np.dot(X_tr.values, X_ts.values[review_loc, :].T)

In [47]:
print(cos_sim)

[[0.08319411]
 [0.08903182]
 [0.08450808]
 ...
 [0.07497808]
 [0.07954401]
 [0.11991222]]


In [48]:
len(cos_sim)

25000

In [49]:
print(max(cos_sim))

[0.24348834]


In [50]:
doc_tr = X_tr.index[np.argmax(cos_sim)]

In [ ]:
print(review, doc_tr)
# Imprime qué reseña de entrenamiento es la más similar a la seleccionada de test.

4_10.txt 3722_10.txt
